# Weather Hazard Category · Dummy Classifier

**36120 Advanced Machine Learning Application** · Assessment Task 2 · Weather Intelligence (Open-Meteo, Sydney)

*Experiment notebook · predicts WHC exactly +7 days ahead*

This notebook trains and evaluates the **Dummy Classifier** model for the **Weather Hazard Category (WHC)**, using the leakage-audited feature matrix produced by the preprocessing notebook.

### Student details

| | |
|:--|:--|
| **Student name** | Juan Vargas T. |
| **Student ID** | 25945320 |
| **Subject** | 36120 Advanced Machine Learning Application |
| **Assessment** | AT2 · Weather Intelligence |
| **GitHub** | [Sebas102507](https://github.com/Sebas102507) |

## Overview

### Hypothesis

`DummyClassifier(strategy="most_frequent")` ignores the features and predicts the most common
training class every day. It is the correct **null model**: every other hazard experiment must
beat it.

The hypothesis governs how every other hazard notebook is read: **this model will have the
highest accuracy in the project and close to the lowest macro F1.** If that holds, accuracy is
disqualified as a selection metric for this target by measurement, not by assertion.

It also defines what a *collapsed* model looks like: any classifier whose confusion matrix
resembles this one has fallen back to the majority class, whatever it cost to train.

### Design

| Choice | Value | Reason |
|:--|:--|:--|
| Algorithm | `DummyClassifier(strategy="most_frequent")` | sklearn's standard majority-class null model |
| Features | Received, then ignored | The rule does not depend on `X` |
| Scaling | None | A class-count rule is unaffected by feature scale |
| Strategy | One classifier for the single +7 day horizon | Matches the deployed artefact layout |
| Selection | None: `default_params` is used as is | A null model has nothing worth tuning (section 5) |
| Objective | validation **macro F1** (higher is better) | Used for every comparison |

### Workflow

All ten experiment notebooks run the same code from `nucleus_ml.weather.Experiment`, so their results are directly comparable. **Only the model specification in section 1 differs between notebooks**, and because this is a null model the tuning steps of the other notebooks are left out: the final model uses `default_params` directly.

| # | Step | Purpose |
|:-:|:--|:--|
| 1 | Model specification | The only algorithm-specific cell |
| 2 | Data | Training and validation splits (the test split is never loaded here) |
| 3 | Baselines | The floor every model must clear |
| 4 | Feature-family ablation | What each block of engineered features contributes |
| 5 | Fixed configuration | Use `default_params` as is: no search |
| 6 | Final model | Final model against the baselines |
| 7 | Feature importance | Which features the model relies on |
| 8 | Generalisation check | Training vs validation performance |
| 9 | Save artefacts | Model bundle + `/model-metadata` document |
| 10 | Conclusions | What to carry into the report |

---
## 0. Setup

Imports, the experiment's identity and the task definition. Everything else comes from the project package, so no helper is redefined in this notebook.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.dummy import DummyClassifier

from nucleus_ml import weather as wx

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)

TASK = "weather_hazard"
EXPERIMENT = "whc_dummy"
ALGORITHM = "Dummy Classifier"

task = wx.load_task(TASK)
print(f"{task.target_name} | {ALGORITHM} | {len(task.feature_columns)} features | "
      f"targets: {task.target_columns}")

---
## 1. Model specification

This is the **only algorithm-specific cell**. It creates the experiment from the estimator class, which the package builds (seeding it and, where requested, scaling the features), so no helper function is needed. It also sets the configuration the model always uses (`default_params`).

The model has a single setting, `strategy="most_frequent"`, which is used as is. A majority-class
rule has no parameters worth tuning, and fixing it keeps the floor a property of the labels
rather than of a search.

In [ ]:
exp = wx.Experiment(
    task,
    name=EXPERIMENT,
    algorithm=ALGORITHM,
    estimator=DummyClassifier,
    default_params={"strategy": "most_frequent"},  # used as is: no search in this notebook
)

---
## 2. Data

Only the **training** and **validation** splits are loaded. The test split is
reserved for a single evaluation in the comparison notebook, once one model has been
selected, so nothing in this notebook can be influenced by it.

> The preprocessing notebook already applied the chronological split, the warm-up
> filter and an embargo at each boundary. Nothing is re-split or shuffled here.

In [ ]:
exp.data_summary()

---
## 3. Baselines

Three naive forecasts, all fitted on the training split only:

| Baseline | Rule | What it knows |
|:--|:--|:--|
| **Majority class** | Always predict the most common class | Nothing but the class balance |
| **Persistence** | The class in 7 days equals today's class (from today's WHI) | The current weather |
| **Seasonal mode** | The most common class for that month | The season |

The majority-class rule scores a high *accuracy* but a low *macro F1*, which is exactly why
accuracy is not used to select models for this imbalanced target.

In [ ]:
exp.run_baselines()

---
## 4. Feature-family ablation

Feature families are added **one at a time**, starting from persistence alone (a single column), and the model is refitted at each step. This measures what each block of engineering contributes instead of assuming it. `ablation_params` are used throughout so the differences come from the features, not from tuning. Scores are on validation (macro F1).

> A null model ignores its inputs, so every row should score the same: this confirms the model does not use `X`.

In [ ]:
ablation = exp.run_ablation()
plt.show()
ablation

---
## 5. Fixed configuration

No hyperparameter search is run. The final model uses `default_params` exactly
as defined in section 1. A null model has no settings whose choice would be
informative, and keeping it fixed means the floor it sets is a property of the
labels, not of a search.

In [ ]:
exp.use_default_params()

---
## 6. Final model

The model is fitted with `default_params` and compared with every baseline (it appears as `default`). `gain_of_final_model` is positive when the final model is better than that row.

Read the **confusion matrix** as well as the headline score: a model that almost never predicts
the rare classes has not learned to flag hazardous days, whatever its macro F1. The printed
boundary summary shows how many errors sit within a few WHI points of a class threshold; those are
threshold effects rather than genuine misses.

In [ ]:
exp.finalise()

In [ ]:
exp.plot_validation()
plt.show()

---
## 7. Feature importance

A dummy model ignores the features, so there is no importance to report.

> Correlated features (a variable, its lags and its rolling means) share importance, so a feature that looks unimportant on its own may belong to an essential block. The ablation in section 4 is the better guide to what can be removed.

In [ ]:
family_importance = exp.importance()
plt.show()
family_importance

---
## 8. Generalisation check

Training and validation scores side by side. A small gap means the model
generalises; a large gap means it has memorised patterns in the training years that
do not carry over to 2024.

In [ ]:
exp.generalisation()

---
## 9. Save artefacts

The fitted model(s) are written to
`models/weather_hazard/experiments/whc_dummy/` with:

| File | Contents |
|:--|:--|
| `whc_dummy_h<horizon>.joblib` | One fitted model per horizon |
| `metadata.json` | Target, algorithm, horizon, features, importance, hyperparameters, metrics — the `/model-metadata` document |
| `manifest.json` | File list and the feature column order the API must use |

> The feature column **order** is recorded because models receive an unnamed array
> at prediction time and would silently give wrong answers if the columns were
> permuted.

In [ ]:
exp.export()

---
## 10. Conclusions

- **This is the floor, not the bar.** Persistence and the seasonal mode use information and should
  beat it; the gap measures what current weather and the calendar are each worth.
- **The metric argument, measured.** Report this model's accuracy and macro F1 side by side
  (section 6). One near the majority share and one near the floor is the clearest
  justification for not optimising accuracy on this target.
- **The collapse test.** Compare the other four confusion matrices with this one before reporting
  any of them as successful.

> The test split has still not been used. It is scored exactly once, in the `whc_comparison` notebook, after one model has been selected on validation performance across all five WHC experiments.